# Data preprocessing exercise solutions

Compare the reasoning and leakage controls, not only the exact spelling of the code.

In [ ]:
import re
import unicodedata

import numpy as np
import pandas as pd

data = pd.DataFrame({
    "response_id": ["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8"],
    "learner_id": ["L1", "L1", "L2", "L2", "L3", "L3", "L4", "L4"],
    "text": [
        "من كتاب دارم", "من کتاب دارم", "فارسی جالب است", "فارسی سخت نیست",
        "سلام 😊", "تمرین خیلییی سخت بود", "به دانشگاه می‌روم", "زبان را دوست دارم",
    ],
    "age": [22.0, 22.0, np.nan, np.nan, 31.0, 31.0, 27.0, 27.0],
    "tokens": [3, 3, 3, 3, 2, 4, 3, 4],
    "task_type": ["free", "free", "picture", "picture", "social", "social", "free", None],
    "needs_review": [0, 0, 0, 1, 0, 1, 0, 0],
})

data

## Solution 1 — missingness audit

In [ ]:
missing_report = pd.DataFrame({
    "missing_n": data.isna().sum(),
    "missing_pct": data.isna().mean().mul(100).round(1),
    "dtype": data.dtypes.astype(str),
    "unique_n": data.nunique(dropna=False),
})
missing_report

## Solution 2 — feature contract

In [ ]:
X = data.drop(columns=["response_id", "learner_id", "needs_review"])
y = data["needs_review"]
print(X.columns.tolist())
print(y.tolist())

## Solution 3 — stratified split

In [ ]:
from sklearn.model_selection import train_test_split

independent = pd.DataFrame({"value": np.arange(50), "label": [0] * 40 + [1] * 10})
independent_train, independent_test = train_test_split(
    independent,
    test_size=0.20,
    random_state=42,
    stratify=independent["label"],
)
print(independent_train["label"].mean(), independent_test["label"].mean())

## Solution 4 — group split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=42)
train_idx, test_idx = next(splitter.split(data, groups=data["learner_id"]))
group_train = data.iloc[train_idx]
group_test = data.iloc[test_idx]

assert set(group_train["learner_id"]).isdisjoint(group_test["learner_id"])
print(sorted(group_train["learner_id"].unique()))
print(sorted(group_test["learner_id"].unique()))

## Solution 5 — train-fitted imputation

In [ ]:
from sklearn.impute import SimpleImputer

age_train = group_train[["age"]]
age_test = group_test[["age"]]
age_imputer = SimpleImputer(strategy="median", add_indicator=True)
age_train_ready = age_imputer.fit_transform(age_train)
age_test_ready = age_imputer.transform(age_test)
print(age_imputer.statistics_)
print(age_train_ready)
print(age_test_ready)

## Solution 6 — unknown-safe one-hot encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

category_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
category_encoder.fit(pd.DataFrame({"task_type": ["free", "picture", "social"]}))
future_category = category_encoder.transform(pd.DataFrame({"task_type": ["interview"]}))
print(category_encoder.get_feature_names_out().tolist())
print(future_category)

## Solution 7 — robust scaling

In [ ]:
from sklearn.preprocessing import RobustScaler

token_scaler = RobustScaler()
token_scaler.fit(group_train[["tokens"]])
future_tokens_scaled = token_scaler.transform(pd.DataFrame({"tokens": [5, 100]}))
print(future_tokens_scaled)

## Solution 8 — mixed-column preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
mixed_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, ["age", "tokens"]),
    ("categorical", categorical_pipeline, ["task_type"]),
])
X_ready = mixed_preprocessor.fit_transform(data)
print(X_ready.shape)
print(mixed_preprocessor.get_feature_names_out().tolist())

## Solution 9 — conservative Persian normalization

In [ ]:
ARABIC_TO_PERSIAN = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک"})
INVISIBLE_PATTERN = re.compile(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069]")
SPACE_PATTERN = re.compile(r"[ \t\r\f\v]+")

def normalize_persian(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = text.translate(ARABIC_TO_PERSIAN)
    text = INVISIBLE_PATTERN.sub("", text)
    text = re.sub(r"\s*\u200c\s*", "\u200c", text)
    return SPACE_PATTERN.sub(" ", text).strip()

print(normalize_persian("  من كتاب فارسي را دوست دارم 😊  "))

## Solution 10 — train-fitted TF–IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

normalized = data["text"].map(normalize_persian)
vectorizer = TfidfVectorizer(
    preprocessor=normalize_persian,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
)
X_text_train = vectorizer.fit_transform(normalized.iloc[:6])
X_text_test = vectorizer.transform(normalized.iloc[6:])
print(X_text_train.shape, X_text_test.shape)
print(vectorizer.get_feature_names_out()[:10].tolist())

## Challenge solution — leakage review

In [ ]:
valid_features = ["text", "task_type", "age"]
split_or_audit = ["response_id", "learner_id"]
forbidden = ["annotated_error_count", "annotator_final_confidence", "needs_review"]
print(valid_features, split_or_audit, forbidden)